In [ ]:
import gdown, zipfile, os

file_id = "1yDOCT8WbfaHjXR79wp2yT7-rLNq0cNWD"
zip_path = "/content/box-factory-dataset.zip"
extract_path = "/content/dataset"

if not os.path.exists(extract_path):
    print("📥 در حال دانلود دیتاست...")
    gdown.download(f"https://drive.google.com/uc?id={file_id}", zip_path, quiet=False)
    print("📦 در حال استخراج...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print("✅ استخراج کامل شد.")
else:
    print("✅ دیتاست از قبل موجود است.")

for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    if level < 2:
        print('  ' * level + os.path.basename(root) + '/')
    if level >= 2:
        break

📥 در حال دانلود دیتاست...


Downloading...
From (original): https://drive.google.com/uc?id=1yDOCT8WbfaHjXR79wp2yT7-rLNq0cNWD
From (redirected): https://drive.google.com/uc?id=1yDOCT8WbfaHjXR79wp2yT7-rLNq0cNWD&confirm=t&uuid=fe241cf7-baad-44bc-88ad-e5ef4e03f58d
To: /content/box-factory-dataset.zip
100%|██████████| 107M/107M [00:02<00:00, 38.4MB/s]


📦 در حال استخراج...
✅ استخراج کامل شد.
dataset/
  box-factory-dataset/


In [3]:
# ۳. ساخت خودکار data.yaml با مسیر مطلق
import yaml

class_names = ["box"]  # نام کلاس‌ها را مطابق دیتاست تنظیم کنید

data_yaml_content = {
    'path': extract_path,
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: name for i, name in enumerate(class_names)}
}

yaml_save_path = os.path.join(extract_path, 'data.yaml')
with open(yaml_save_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f"✅ data.yaml ساخته شد: {yaml_save_path}")
print(open(yaml_save_path).read())

✅ data.yaml ساخته شد: /content/dataset/data.yaml
names:
  0: box
path: /content/dataset
train: images/train
val: images/val



In [8]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else 'cpu'
print(f"🚀 شروع آموزش روی: {'GPU' if device == 0 else 'CPU'}")

model = YOLO("yolo11n.pt")
model.train(
    data=yaml_path_out,      # ← مسیر data_fixed.yaml
    epochs=60,
    imgsz=640,
    batch=16,
    device=device,
    project="weights",
    name="box_detector",
    patience=15,
    exist_ok=True,
)
print("✅ آموزش تمام شد. وزن‌ها: weights/box_detector/weights/best.pt")

🚀 شروع آموزش روی: GPU
Ultralytics 8.4.199 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/box-factory-dataset/train/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=box_detec

In [11]:
import os, glob, json, csv

RUN_DIR = "/content/runs/detect/weights/box_detector"
TRAIN_DIR = "/content/dataset/box-factory-dataset/train"

print("=" * 60)
print("۱. متریک‌های نهایی آموزش (results.csv)")
print("=" * 60)
csv_path = os.path.join(RUN_DIR, "results.csv")
if os.path.exists(csv_path):
    with open(csv_path) as f:
        lines = f.readlines()
    # هدر + ۳ ردیف آخر
    print(lines[0].strip())
    for line in lines[-3:]:
        print(line.strip())
else:
    print("❌ results.csv پیدا نشد")

print("\n" + "=" * 60)
print("۲. اطلاعات دیتاست (summary.json)")
print("=" * 60)
summary_path = os.path.join(TRAIN_DIR, "summary.json")
if os.path.exists(summary_path):
    data = json.load(open(summary_path))
    print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])
else:
    print("❌ summary.json پیدا نشد")

print("\n" + "=" * 60)
print("۳. فایل data_fixed.yaml")
print("=" * 60)
yaml_path = os.path.join(TRAIN_DIR, "data_fixed.yaml")
if os.path.exists(yaml_path):
    print(open(yaml_path).read())
else:
    print("❌ پیدا نشد")

print("\n" + "=" * 60)
print("۴. لیست فایل‌های خروجی آموزش")
print("=" * 60)
if os.path.exists(RUN_DIR):
    for f in sorted(os.listdir(RUN_DIR)):
        p = os.path.join(RUN_DIR, f)
        size = os.path.getsize(p) / 1024
        print(f"  {f}  ({size:.1f} KB)")
else:
    print("❌ پوشه خروجی پیدا نشد")

print("\n" + "=" * 60)
print("۵. تعداد تصاویر train / val")
print("=" * 60)
for split in ["train", "val"]:
    img_dir = os.path.join(TRAIN_DIR, "images", split)
    lbl_dir = os.path.join(TRAIN_DIR, "labels", split)
    n_img = len(glob.glob(os.path.join(img_dir, "*"))) if os.path.exists(img_dir) else 0
    n_lbl = len(glob.glob(os.path.join(lbl_dir, "*.txt"))) if os.path.exists(lbl_dir) else 0
    print(f"  {split}: {n_img} تصویر، {n_lbl} برچسب")

۱. متریک‌های نهایی آموزش (results.csv)
epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
58,108.932,0.42276,0.39836,0.81718,0.99683,0.9748,0.995,0.92824,0.38697,0.33616,0.85724,9.91865e-05,9.91865e-05,9.91865e-05
59,110.427,0.43362,0.43126,0.83652,0.99683,0.97504,0.995,0.93332,0.38659,0.33737,0.85689,7.1681e-05,7.1681e-05,7.1681e-05
60,111.928,0.41169,0.38925,0.82635,0.99744,0.97766,0.995,0.93158,0.38495,0.32887,0.85671,4.41755e-05,4.41755e-05,4.41755e-05

۲. اطلاعات دیتاست (summary.json)
{
  "images": 100,
  "train_images": 85,
  "val_images": 15,
  "objects": 352,
  "class_counts": {
    "healthy_box": 205,
    "defective_box": 147
  }
}

۳. فایل data_fixed.yaml
names:
  0: healthy_box
  1: defective_box
path: /content/dataset/box-factory-dataset/train
train: images/train
val: images/val


۴. لیست فایل‌های خروجی آموزش
  BoxF1_curve.png  (124.5 

In [26]:
import os, glob, cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm

TRAINED = "/content/runs/detect/weights/box_detector/weights/best.pt"
VIDEOS_DIR = "/content/dataset/box-factory-dataset/test/videos/raw"
OUTPUT_CSV = "/content/submission.csv"

CONF_THRESHOLD = 0.4
ENTRY_X_RATIO = 0.15     # خط ورود در ۱۵٪ عرض تصویر (نزدیک درب آبی)
MAX_BOXES = 6
MIN_HITS = 2
MAX_LOST = 20
DIST_THR = 50            # فاصله مجاز بین فریم‌ها (کمتر از عرض جعبه)

# ============================================================
# ردیاب مرکز-محور
# ============================================================
class CenterTracker:
    def __init__(self, dist_thr=DIST_THR, max_lost=MAX_LOST):
        self.tracks = {}
        self.next_id = 0
        self.dist_thr = dist_thr
        self.max_lost = max_lost

    def update(self, detections):
        used = set()
        for cx, cy, cls in detections:
            best_id, best_dist = None, self.dist_thr
            for tid, tr in self.tracks.items():
                if tid in used:
                    continue
                # فقط تطبیق با مسیرهای فعال یا خیلی تازه
                if tr["lost"] > 3:
                    continue
                dist = abs(tr["cx"] - cx)
                if dist < best_dist:
                    best_dist, best_id = dist, tid
            if best_id is not None:
                tr = self.tracks[best_id]
                tr["cx"] = cx
                tr["cy"] = cy
                tr["cls"] = cls
                tr["hits"] += 1
                tr["lost"] = 0
                used.add(best_id)
            else:
                self.tracks[self.next_id] = {
                    "cx": cx, "cy": cy, "cls": cls,
                    "hits": 1, "lost": 0, "counted": False,
                    "first_cx": cx,
                }
                used.add(self.next_id)
                self.next_id += 1

        to_del = [tid for tid, tr in self.tracks.items()
                  if tid not in used and tr["lost"] > self.max_lost]
        for tid in to_del:
            del self.tracks[tid]
        for tid, tr in self.tracks.items():
            if tid not in used:
                tr["lost"] += 1

# ============================================================
# پردازش ویدیو
# ============================================================
def process_video(video_path, model):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    entry_x = int(w * ENTRY_X_RATIO)

    tracker = CenterTracker()
    counted = []          # (frame_idx, cls)
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # هر ۳ فریم یک‌بار برای سرعت
        if frame_idx % 3 != 0:
            frame_idx += 1
            continue

        results = model.predict(
            frame, conf=CONF_THRESHOLD, iou=0.45, verbose=False
        )[0]

        detections = []
        if results.boxes is not None and len(results.boxes) > 0:
            for box in results.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                cls = int(box.cls[0].item())
                detections.append((cx, cy, cls))

        # مرتب‌سازی چپ به راست برای تطبیق بهتر
        detections.sort(key=lambda d: d[0])
        tracker.update(detections)

        # شمارش: هر جعبه‌ای که برای اولین بار از خط ورود عبور کند
        for tid, tr in tracker.tracks.items():
            if tr["counted"]:
                continue
            if tr["hits"] < MIN_HITS:
                continue
            # شرط ورود: مرکز x از خط ورود (به سمت راست) رد شده باشد
            if tr["cx"] > entry_x and tr["first_cx"] <= entry_x + 50:
                tr["counted"] = True
                counted.append((frame_idx, tr["cls"]))

        frame_idx += 1

    cap.release()
    counted.sort(key=lambda x: x[0])
    return [c for _, c in counted]

# ============================================================
# اجرا
# ============================================================
print("📦 بارگذاری مدل...")
model = YOLO(TRAINED)

videos = sorted(glob.glob(os.path.join(VIDEOS_DIR, "*.webm")) +
                glob.glob(os.path.join(VIDEOS_DIR, "*.mp4")))
print(f"📹 {len(videos)} ویدیو")

rows = []
counts = []
for vp in tqdm(videos, desc="Processing"):
    vid_id = os.path.splitext(os.path.basename(vp))[0]
    try:
        statuses = process_video(vp, model)
    except Exception as e:
        print(f"⚠️ خطا در {vid_id}: {e}")
        statuses = []

    statuses = statuses[:MAX_BOXES]
    counts.append(len(statuses))
    row = [vid_id] + statuses + [-1] * (MAX_BOXES - len(statuses))
    rows.append(row)

df = pd.DataFrame(rows, columns=["video_id", "1", "2", "3", "4", "5", "6"])
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ {OUTPUT_CSV} ساخته شد")
print(df.to_string(index=False))

print(f"\n📊 توزیع تعداد جعبه:")
import collections
c = collections.Counter(counts)
for k in sorted(c):
    print(f"  {k} جعبه: {c[k]} ویدیو")

📦 بارگذاری مدل...
📹 30 ویدیو


Processing: 100%|██████████| 30/30 [01:02<00:00,  2.10s/it]


✅ /content/submission.csv ساخته شد
            video_id  1  2  3  4  5  6
factory_test_001_raw  0  1  0  1  0  1
factory_test_002_raw  0  1  1  1  1  0
factory_test_003_raw  0  1  0  0  0  1
factory_test_004_raw  0  0  1  1  1  0
factory_test_005_raw  0  1  1  1  0  0
factory_test_006_raw  0  1  0  1  0  0
factory_test_007_raw  0  1  0  0  1  1
factory_test_008_raw  0  1  0  1  1  0
factory_test_009_raw  0  1  0  1  1  1
factory_test_010_raw  0  1  1  0  0  1
factory_test_011_raw  0  1  1  1  0  0
factory_test_012_raw  1  1  0  0  1  1
factory_test_013_raw  0  1  0  1  1  1
factory_test_014_raw  0  1  0  0  1  0
factory_test_015_raw  0  1  0  1  0  1
factory_test_016_raw  0  1  1  1  0  0
factory_test_017_raw  0  1  0  1  1  1
factory_test_018_raw  0  1  0  0  0  0
factory_test_019_raw  0  1  1  0  0  0
factory_test_020_raw  0  1  1  0  1  1
factory_test_021_raw  0  0  0  1  0  1
factory_test_022_raw  0  1  1  0  0  1
factory_test_023_raw  0  1  0  1  0  1
factory_test_024_raw  0  1  

In [28]:
import os, shutil, zipfile

# مسیر فایل‌ها
notebook_path = "/content/notebook.ipynb"   # نام فایلی که آپلود کردید
csv_path = "/content/submission.csv"         # فایل خروجی شما
zip_output = "/content/submission_correct.zip"

# بررسی وجود فایل‌ها
print("notebook.ipynb موجود:", os.path.exists(notebook_path))
print("submission.csv موجود:", os.path.exists(csv_path))

if not os.path.exists(notebook_path):
    print("❌ فایل notebook.ipynb را آپلود نکرده‌اید!")
elif not os.path.exists(csv_path):
    print("❌ فایل submission.csv را نساخته‌اید!")
else:
    # ساخت zip با ساختار صحیح (فایل‌ها در ریشه zip، نه داخل پوشه)
    with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(notebook_path, arcname="notebook.ipynb")
        zf.write(csv_path, arcname="submission.csv")

    print(f"\n✅ {zip_output} ساخته شد")
    # نمایش محتوای zip برای تأیید
    with zipfile.ZipFile(zip_output, 'r') as zf:
        print("محتوای zip:", zf.namelist())

notebook.ipynb موجود: False
submission.csv موجود: True
❌ فایل notebook.ipynb را آپلود نکرده‌اید!


In [24]:
import os, glob, cv2
import numpy as np
from ultralytics import YOLO

TRAINED = "/content/runs/detect/weights/box_detector/weights/best.pt"
VIDEOS_DIR = "/content/dataset/box-factory-dataset/test/videos/raw"
model = YOLO(TRAINED)
videos = sorted(glob.glob(os.path.join(VIDEOS_DIR, "*.webm")))

# برای هر ویدیو، تمام مراکز x را جمع کن و "خوشه" های مجزا را بشمار
print(f"{'ویدیو':<30} {'تشخیص':>6} {'خوشه x':>8} {'x_min':>6} {'x_max':>6}")
print("-" * 70)

for vp in videos[:10]:
    cap = cv2.VideoCapture(vp)
    all_cx = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if frame_idx % 3 == 0:
            r = model.predict(frame, conf=0.3, verbose=False)[0]
            if r.boxes is not None:
                for b in r.boxes:
                    x1, y1, x2, y2 = b.xyxy[0].tolist()
                    all_cx.append((x1+x2)/2)
        frame_idx += 1
    cap.release()

    if not all_cx:
        print(f"{os.path.basename(vp):<30} 0")
        continue

    # خوشه‌بندی ساده: هر مرکز x که از خوشه قبلی بیش از ۴۰ پیکسل فاصله دارد، خوشه جدید
    all_cx_sorted = sorted(all_cx)
    clusters = [all_cx_sorted[0]]
    for x in all_cx_sorted[1:]:
        if x - clusters[-1] > 40:
            clusters.append(x)

    vid = os.path.basename(vp)[:28]
    print(f"{vid:<30} {len(all_cx):>6} {len(clusters):>8} {min(all_cx):>6.0f} {max(all_cx):>6.0f}")

print("\nتفسیر: ستون 'خوشه x' تعداد تقریبی جعبه‌های مجزا در ویدیو است.")
print("اگر خوشه x برابر ۲-۴ بود ولی submission ۶ نشان می‌دهد → بیش‌شماری داریم.")

ویدیو                           تشخیص   خوشه x  x_min  x_max
----------------------------------------------------------------------
factory_test_001_raw.webm         298       25    187   1268
factory_test_002_raw.webm         265       26    183   1269
factory_test_003_raw.webm         292       26    191   1265
factory_test_004_raw.webm         336       25    177   1269
factory_test_005_raw.webm         300       26    186   1265
factory_test_006_raw.webm         260       26    178   1273
factory_test_007_raw.webm         341       26    178   1273
factory_test_008_raw.webm         320       27    179   1271
factory_test_009_raw.webm         314       25    179   1269
factory_test_010_raw.webm         314       25    185   1272

تفسیر: ستون 'خوشه x' تعداد تقریبی جعبه‌های مجزا در ویدیو است.
اگر خوشه x برابر ۲-۴ بود ولی submission ۶ نشان می‌دهد → بیش‌شماری داریم.


In [29]:
import os, glob, cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm

TRAINED = "/content/runs/detect/weights/box_detector/weights/best.pt"
VIDEOS_DIR = "/content/dataset/box-factory-dataset/test/videos/raw"
OUTPUT_CSV = "/content/submission.csv"

CONF_THRESHOLD = 0.4
EXIT_X_RATIO = 0.90    # خط خروج: ۹۰٪ عرض تصویر (نزدیک لبه راست)
MAX_BOXES = 6
MIN_HITS = 3

# ============================================================
# ردیاب ساده بر اساس فاصله افقی
# ============================================================
class Tracker:
    def __init__(self, dist_thr=60, max_lost=15):
        self.tracks = {}
        self.next_id = 0
        self.dist_thr = dist_thr
        self.max_lost = max_lost

    def update(self, detections):
        used = set()
        for cx, cy, cls in detections:
            best_id, best_dist = None, self.dist_thr
            for tid, tr in self.tracks.items():
                if tid in used or tr["lost"] > 2:
                    continue
                d = abs(tr["cx"] - cx)
                if d < best_dist:
                    best_dist, best_id = d, tid
            if best_id is not None:
                tr = self.tracks[best_id]
                tr["cx"], tr["cy"], tr["cls"] = cx, cy, cls
                tr["hits"] += 1
                tr["lost"] = 0
                used.add(best_id)
            else:
                self.tracks[self.next_id] = {
                    "cx": cx, "cy": cy, "cls": cls,
                    "hits": 1, "lost": 0, "counted": False,
                    "max_cx": cx,
                }
                used.add(self.next_id)
                self.next_id += 1
        to_del = [tid for tid, tr in self.tracks.items()
                  if tid not in used and tr["lost"] > self.max_lost]
        for tid in to_del:
            del self.tracks[tid]
        for tid, tr in self.tracks.items():
            if tid not in used:
                tr["lost"] += 1

# ============================================================
# پردازش ویدیو: شمارش در لحظه خروج از خط راست
# ============================================================
def process_video(video_path, model):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    exit_x = int(w * EXIT_X_RATIO)

    tracker = Tracker()
    counted = []   # (frame_idx, cls)
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % 2 != 0:
            frame_idx += 1
            continue

        results = model.predict(
            frame, conf=CONF_THRESHOLD, iou=0.45, verbose=False
        )[0]

        detections = []
        if results.boxes is not None:
            for box in results.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx, cy = (x1+x2)/2, (y1+y2)/2
                detections.append((cx, cy, int(box.cls[0].item())))
        detections.sort(key=lambda d: d[0])   # چپ به راست
        tracker.update(detections)

        for tid, tr in tracker.tracks.items():
            if tr["counted"]:
                continue
            if tr["hits"] < MIN_HITS:
                continue
            tr["max_cx"] = max(tr["max_cx"], tr["cx"])
            # اگر جعبه به خط خروج رسید
            if tr["cx"] >= exit_x:
                tr["counted"] = True
                counted.append((frame_idx, tr["cls"]))

        frame_idx += 1

    cap.release()
    counted.sort(key=lambda x: x[0])
    return [c for _, c in counted]

# ============================================================
# اجرا
# ============================================================
print("📦 بارگذاری مدل...")
model = YOLO(TRAINED)
videos = sorted(glob.glob(os.path.join(VIDEOS_DIR, "*.webm")) +
                glob.glob(os.path.join(VIDEOS_DIR, "*.mp4")))
print(f"📹 {len(videos)} ویدیو\n")

rows = []
counts = []
for vp in tqdm(videos, desc="Processing"):
    vid_id = os.path.splitext(os.path.basename(vp))[0]
    try:
        statuses = process_video(vp, model)
    except Exception as e:
        print(f"⚠️ خطا در {vid_id}: {e}")
        statuses = []
    statuses = statuses[:MAX_BOXES]
    counts.append(len(statuses))
    rows.append([vid_id] + statuses + [-1] * (MAX_BOXES - len(statuses)))

df = pd.DataFrame(rows, columns=["video_id", "1", "2", "3", "4", "5", "6"])
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ {OUTPUT_CSV} ساخته شد")
print(df.to_string(index=False))

print(f"\n📊 توزیع تعداد:")
from collections import Counter
c = Counter(counts)
for k in sorted(c):
    print(f"  {k} جعبه: {c[k]} ویدیو")

📦 بارگذاری مدل...
📹 30 ویدیو



Processing: 100%|██████████| 30/30 [01:20<00:00,  2.68s/it]


✅ /content/submission.csv ساخته شد
            video_id  1  2  3  4  5  6
factory_test_001_raw  0  1  0  0  1  0
factory_test_002_raw  0  0  1  0  1  0
factory_test_003_raw  0  1  0  0  0  1
factory_test_004_raw  0  1  0  1  0  1
factory_test_005_raw  0  1  0  0  1  0
factory_test_006_raw  0  1  0  1  0  0
factory_test_007_raw  0  1  0  1  0  1
factory_test_008_raw  0  0  1  0  1  0
factory_test_009_raw  0  1  0  0  1  1
factory_test_010_raw  0  1  1  0  0  1
factory_test_011_raw  0  0  1  1  1  1
factory_test_012_raw  0  1  1  0  1  0
factory_test_013_raw  0  0  1  0  0  0
factory_test_014_raw  0  1  0  0  1  0
factory_test_015_raw  0  0  0  0  1  0
factory_test_016_raw  0  1  0  0  0  0
factory_test_017_raw  0  1  0  0  1  0
factory_test_018_raw  0  1  0  0  0  0
factory_test_019_raw  0  1  0  0  0  0
factory_test_020_raw  0  1  0  0  1  0
factory_test_021_raw  0  0  0  0  1  1
factory_test_022_raw  0  1  0  1  1  0
factory_test_023_raw  0  1  0  1  1  0
factory_test_024_raw  0  1  

In [32]:
import os, glob, cv2, pandas as pd, numpy as np
from ultralytics import YOLO
from tqdm import tqdm

# ============================================================
# تنظیمات
# ============================================================
MODEL_PATH   = "weights/box_detector/weights/best.pt"
VIDEOS_DIR   = "test/videos/raw"
OUT_CSV      = "/content/submission.csv"
CONF_THR     = 0.25
IOU_THR      = 0.45
MIN_HITS     = 3
LINE_Y_RATIO = 0.55
MAX_BOXES    = 6          # حداکثر ۶ جعبه در هر ویدیو
DEVICE       = 0 if __import__('torch').cuda.is_available() else 'cpu'

# ============================================================
# ردیاب ساده مبتنی بر IoU
# ============================================================
class SimpleTracker:
    def __init__(self, iou_thr=0.3, max_lost=30):
        self.tracks, self.next_id = {}, 0
        self.iou_thr, self.max_lost = iou_thr, max_lost

    def iou(self, a, b):
        ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
        x1, y1 = max(ax1, bx1), max(ay1, by1)
        x2, y2 = min(ax2, bx2), min(ay2, by2)
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        ua = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter + 1e-6
        return inter / ua

    def update(self, detections):
        used = set()
        for box, cls in detections:
            best_id, best_iou = None, self.iou_thr
            for tid, tr in self.tracks.items():
                if tid in used: continue
                v = self.iou(box, tr["bbox"])
                if v > best_iou: best_iou, best_id = v, tid
            if best_id is not None:
                tr = self.tracks[best_id]
                tr.update(bbox=box, cls=cls, hits=tr["hits"]+1, lost=0)
                used.add(best_id)
            else:
                self.tracks[self.next_id] = {
                    "bbox": box, "cls": cls, "hits": 1, "lost": 0, "counted": False
                }
                used.add(self.next_id); self.next_id += 1

        for tid in [t for t, tr in self.tracks.items()
                    if t not in used and tr["lost"] > self.max_lost]:
            del self.tracks[tid]
        for tid, tr in self.tracks.items():
            if tid not in used: tr["lost"] += 1


# ============================================================
# پردازش یک ویدیو
# ============================================================
def process_video(video_path, model):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    line_y = int(h * LINE_Y_RATIO)

    tracker = SimpleTracker()
    counted, prev_y = [], {}

    while True:
        ret, frame = cap.read()
        if not ret: break

        r = model.predict(frame, conf=CONF_THR, iou=IOU_THR,
                          device=DEVICE, verbose=False)[0]
        dets = []
        if r.boxes is not None and len(r.boxes) > 0:
            for b in r.boxes:
                x1, y1, x2, y2 = b.xyxy[0].tolist()
                dets.append(((x1, y1, x2, y2), int(b.cls[0].item())))

        tracker.update(dets)

        for tid, tr in tracker.tracks.items():
            if tr["counted"] or tr["hits"] < MIN_HITS: continue
            cy = (tr["bbox"][1] + tr["bbox"][3]) / 2
            if tid in prev_y and prev_y[tid] < line_y <= cy:
                tr["counted"] = True
                counted.append(tr["cls"])
            prev_y[tid] = cy

    cap.release()
    return counted


# ============================================================
# اجرا و ساخت CSV
# ============================================================
print("📦 بارگذاری مدل:", MODEL_PATH)
model = YOLO(MODEL_PATH)

videos = sorted(glob.glob(os.path.join(VIDEOS_DIR, "*.webm")))
print(f"📹 {len(videos)} ویدیو پیدا شد")
if not videos:
    print("❌ هیچ ویدیویی پیدا نشد. مسیر VIDEOS_DIR را بررسی کنید.")
else:
    rows = []
    for vp in tqdm(videos, desc="Processing"):
        vid_id = os.path.splitext(os.path.basename(vp))[0]
        try:
            st = process_video(vp, model)
        except Exception as e:
            print(f"⚠️ {vid_id}: {e}")
            st = []
        st = st[:MAX_BOXES]
        rows.append([vid_id] + st + [-1] * (MAX_BOXES - len(st)))

    df = pd.DataFrame(rows, columns=["video_id", "1", "2", "3", "4", "5", "6"])
    df.to_csv(OUT_CSV, index=False)
    print(f"\n✅ ساخته شد: {OUT_CSV}")
    print(df.to_string(index=False))

📦 بارگذاری مدل: weights/box_detector/weights/best.pt


FileNotFoundError: [Errno 2] No such file or directory: 'weights/box_detector/weights/best.pt'

In [43]:
import os, glob, cv2
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

TRAINED = "/content/runs/detect/weights/box_detector/weights/best.pt"
VIDEOS_DIR = "/content/dataset/box-factory-dataset/test/videos/raw"
OUTPUT_CSV = "/content/submission.csv"

CONF_THRESHOLD = 0.35
MAX_BOXES = 6
MIN_HITS = 3

class Tracker:
    def __init__(self, dist_thr=70, max_lost=20):
        self.tracks = {}
        self.next_id = 0
        self.dist_thr = dist_thr
        self.max_lost = max_lost

    def update(self, detections):
        used = set()
        for cx, cy, cls in detections:
            best_id, best_dist = None, self.dist_thr
            for tid, tr in self.tracks.items():
                if tid in used or tr["lost"] > 2: continue
                d = abs(tr["cx"] - cx)
                if d < best_dist: best_dist, best_id = d, tid
            if best_id is not None:
                tr = self.tracks[best_id]
                tr["cx"], tr["cy"] = cx, cy
                tr["hits"] += 1
                tr["lost"] = 0
                used.add(best_id)
            else:
                self.tracks[self.next_id] = {
                    "cx": cx, "cy": cy, "cls": cls, # کلاس در لحظه تولد (ورود) ثبت می‌شود
                    "hits": 1, "lost": 0, "counted": False,
                }
                used.add(self.next_id)
                self.next_id += 1
        to_del = [tid for tid, tr in self.tracks.items() if tid not in used and tr["lost"] > self.max_lost]
        for tid in to_del: del self.tracks[tid]
        for tid, tr in self.tracks.items():
            if tid not in used: tr["lost"] += 1

def process_video(video_path, model):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return []
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

    # ما در لحظه خروج (90%) ثبت می‌کنیم تا ترتیب دقیقاً حفظ شود
    exit_line = int(w * 0.90)

    tracker = Tracker()
    counted = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret: break

        if frame_idx % 2 != 0:
            frame_idx += 1
            continue

        results = model.predict(frame, conf=CONF_THRESHOLD, iou=0.45, verbose=False)[0]
        detections = []
        if results.boxes is not None:
            for box in results.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx, cy = (x1+x2)/2, (y1+y2)/2
                detections.append((cx, cy, int(box.cls[0].item())))

        detections.sort(key=lambda d: d[0])
        tracker.update(detections)

        for tid, tr in tracker.tracks.items():
            if tr["counted"]: continue
            if tr["hits"] < MIN_HITS: continue

            # شمارش زمانی که به خط خروج می‌رسد
            if tr["cx"] >= exit_line:
                tr["counted"] = True
                # کلاسی که در لحظه ورود (وقتی جعبه کاملاً واضح بود) ثبت کردیم را استفاده می‌کنیم
                counted.append((frame_idx, tr["cls"]))

        frame_idx += 1

    cap.release()
    # مرتب‌سازی نهایی بر اساس زمان خروج از کادر (دقیق‌ترین ترتیب)
    counted.sort(key=lambda x: x[0])
    return [c for _, c in counted]

print("📦 بارگذاری مدل...")
model = YOLO(TRAINED)
videos = sorted(glob.glob(os.path.join(VIDEOS_DIR, "*.webm")) + glob.glob(os.path.join(VIDEOS_DIR, "*.mp4")))

rows = []
for vp in tqdm(videos, desc="Processing"):
    vid_id = os.path.splitext(os.path.basename(vp))[0]
    statuses = process_video(vp, model)[:MAX_BOXES]
    rows.append([vid_id] + statuses + [-1] * (MAX_BOXES - len(statuses)))

df = pd.DataFrame(rows, columns=["video_id", "1", "2", "3", "4", "5", "6"])
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ {OUTPUT_CSV} ساخته شد")
print(df.to_string(index=False))

📦 بارگذاری مدل...


Processing: 100%|██████████| 30/30 [01:17<00:00,  2.59s/it]


✅ /content/submission.csv ساخته شد
            video_id  1  2  3  4  5  6
factory_test_001_raw  0  1  0  0  0  1
factory_test_002_raw  0  0  1  1  1  1
factory_test_003_raw  0  1  0  0  0  0
factory_test_004_raw  0  1  0  1  1  1
factory_test_005_raw  0  0  0  0  1  0
factory_test_006_raw  0  1  1  1  0  0
factory_test_007_raw  0  0  0  0  1  0
factory_test_008_raw  0  0  0  0  0  1
factory_test_009_raw  0  1  0  0  0  0
factory_test_010_raw  0  1  0  0  0  1
factory_test_011_raw  0  1  1  1  0  0
factory_test_012_raw  0  0  0  0  0  1
factory_test_013_raw  0  0  1  1  1  1
factory_test_014_raw  1  1  1  0  0  1
factory_test_015_raw  0  1  1  0  1  0
factory_test_016_raw  1  0  0  0  0  0
factory_test_017_raw  0  1  1  0  1  0
factory_test_018_raw  0  0  0  1  1  1
factory_test_019_raw  0  0  0  0  0  0
factory_test_020_raw  0  0  1  0  1  0
factory_test_021_raw  0  0  0  0  0  0
factory_test_022_raw  0  1  1  1  0  0
factory_test_023_raw  0  0  0  0  1  1
factory_test_024_raw  0  0  

In [58]:
import os, yaml, torch
from ultralytics import YOLO

# مسیر فایل yaml دیتاست
BASE = "/content/dataset/box-factory-dataset/train"
yaml_path = os.path.join(BASE, "data_train_final.yaml")

# ساخت فایل yaml مطمئن
data_yaml = {
    'path': BASE,
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: "healthy_box", 1: "defective_box"}
}
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print("🚀 شروع آموزش مدل قدرتمند (YOLO11s) با رزولوشن 800...")
device = 0 if torch.cuda.is_available() else 'cpu'

# استفاده از مدل s (قوی‌تر از n)
model = YOLO("yolo11s.pt")

model.train(
    data=yaml_path,
    epochs=150,             # زمان بیشتر برای یادگیری چهره‌ها
    imgsz=800,              # رزولوشن بالاتر برای دیدن جزئیات
    batch=8,                # کاهش batch برای جلوگیری از پر شدن حافظه GPU
    device=device,
    project="weights",
    name="box_detector_final",
    patience=30,            # اگر ۳۰ دوره بهتر نشد، متوقف شو
    exist_ok=True,
    # تنظیمات Augmentation
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0,
    translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
)
print("✅ آموزش نهایی تمام شد! مدل جدید در مسیر زیر ذخیره شد:")
print("/content/runs/detect/weights/box_detector_final/weights/best.pt")

🚀 شروع آموزش مدل قدرتمند (YOLO11s) با رزولوشن 800...
Ultralytics 8.4.199 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/box-factory-dataset/train/data_train_final.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11s.pt, momentum=0.937, mosai

In [60]:
import os, glob, cv2
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

TRAINED_FINAL = "/content/runs/detect/weights/box_detector_final/weights/best.pt"
VIDEOS_DIR = "/content/dataset/box-factory-dataset/test/videos/raw"
OUTPUT_CSV = "/content/submission.csv"

CONF_THRESHOLD = 0.30
MAX_BOXES = 6

def process_video(video_path, model):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return [0]*MAX_BOXES

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()

    valid_frames = []
    for f_idx in range(0, len(frames), 3):
        frame = frames[f_idx]
        results = model.predict(frame, conf=CONF_THRESHOLD, verbose=False)[0]
        if results.boxes is None: continue

        boxes = []
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2
            cls = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            boxes.append((cx, cls, conf))

        if len(boxes) == MAX_BOXES:
            # 🌟 کلید حل مشکل: مرتب‌سازی از راست به چپ (reverse=True)
            boxes.sort(key=lambda b: b[0], reverse=True)
            valid_frames.append(boxes)

    if valid_frames:
        statuses = []
        for pos in range(MAX_BOXES):
            scores = {0: 0.0, 1: 0.0}
            for boxes in valid_frames:
                cls, conf = boxes[pos][1], boxes[pos][2]
                scores[cls] += conf
            final_cls = max(scores, key=scores.get)
            statuses.append(final_cls)
        return statuses

    return [0]*MAX_BOXES

print("📦 بارگذاری مدل...")
model = YOLO(TRAINED_FINAL)
videos = sorted(glob.glob(os.path.join(VIDEOS_DIR, "*.webm")) + glob.glob(os.path.join(VIDEOS_DIR, "*.mp4")))

rows = []
for vp in tqdm(videos, desc="Processing Videos"):
    vid_id = os.path.splitext(os.path.basename(vp))[0]
    statuses = process_video(vp, model)
    statuses = statuses[:MAX_BOXES]
    while len(statuses) < MAX_BOXES:
        statuses.append(-1)
    rows.append([vid_id] + statuses)

df = pd.DataFrame(rows, columns=["video_id", "1", "2", "3", "4", "5", "6"])
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ {OUTPUT_CSV} ساخته شد")
print(df.to_string(index=False))

📦 بارگذاری مدل...


Processing Videos: 100%|██████████| 30/30 [01:16<00:00,  2.54s/it]


✅ /content/submission.csv ساخته شد
            video_id  1  2  3  4  5  6
factory_test_001_raw  0  1  0  1  0  1
factory_test_002_raw  0  1  1  1  0  0
factory_test_003_raw  0  1  0  0  0  1
factory_test_004_raw  0  1  1  1  1  0
factory_test_005_raw  0  1  1  1  1  0
factory_test_006_raw  0  1  0  1  0  0
factory_test_007_raw  0  1  0  1  1  0
factory_test_008_raw  0  1  0  1  1  0
factory_test_009_raw  0  1  0  1  1  1
factory_test_010_raw  0  1  1  0  0  1
factory_test_011_raw  0  1  1  1  0  0
factory_test_012_raw  0  1  0  1  1  1
factory_test_013_raw  0  1  0  1  1  1
factory_test_014_raw  0  0  0  1  0  0
factory_test_015_raw  0  1  0  1  0  1
factory_test_016_raw  0  1  0  0  0  0
factory_test_017_raw  0  1  0  1  1  1
factory_test_018_raw  0  1  0  0  0  0
factory_test_019_raw  0  1  0  0  0  1
factory_test_020_raw  0  1  1  1  0  0
factory_test_021_raw  0  1  0  0  1  1
factory_test_022_raw  0  1  1  0  0  1
factory_test_023_raw  0  1  0  1  1  1
factory_test_024_raw  0  1  